In [1]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
import re
from functools import reduce
from operator import add

In [2]:
from pyspark.sql import SparkSession
from src.utils.logger import get_logger
import src.utils.config as config 


print(f"DEBUG: Access Key is {config.MINIO_ACCESS_KEY[:3]} + ***") 
print(f"DEBUG: ENDPOINT is {config.MINIO_ENDPOINT}")

# Get the container's hostname dynamically

logger = get_logger(__name__)

def create_spark_session(app_name: str) -> SparkSession:
    """
    Creates and returns a configured Spark session for MinIO.
    """
    logger.info(f"Creating Spark Session: {app_name}")
    
    # This pulls the necessary S3A connectors from Maven Central
    
    

    spark = (   
        SparkSession.builder
        .appName(app_name)
        .master("spark://spark-master:7077")
        .config("spark.driver.host", config.DRIVER_HOST)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.driver.port", config.SPARK_DRIVER_PORT)
        .config("spark.driver.blockManager.port", config.SPARK_BLOCK_MANAGER_PORT)
        .config("spark.sql.shuffle.partitions", "50")
        .config("spark.executor.instances", "1") # adjust based on resources
        .config("spark.executor.cores", "2")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
        #.config("spark.executor.memory", "2g") # adjust based on resources
        #.config("spark.driver.memory", "2g") # adjust based on resources

         # hadoop S3A Configuration
      
        .config("spark.hadoop.fs.s3a.endpoint", config.MINIO_ENDPOINT)
        .config("spark.hadoop.fs.s3a.access.key", config.MINIO_ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", config.MINIO_SECRET_KEY)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(config.MINIO_SECURE).lower())
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # prevent class resolution
        .config("spark.sql.caseSensitive", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.cores.max", "2")
        .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.hadoop.fs.s3a.fast.upload", "true") #performance improvement
        .config("spark.sql.files.maxPartitionBytes", "16777216") #16MB partitions
        #.config("spark.network.timeout", "1200s")
        #.config("spark.rpc.askTimeout", "600s")
        #.config("spark.executor.heartbeatInterval", "120s")
        #.config("spark.hadoop.fs.s3a.connection.timeout", "600000")
        #.config("spark.hadoop.fs.s3a.paging.maximum", "1000")
        
        .getOrCreate()
    )
    # Suppress verbose logs
    spark.sparkContext.setLogLevel("WARN")
    logger.info("Spark session created successfully")
    return spark

[CONFIG] Stage: transform
[CONFIG] Loaded env: /opt/spark-app/env/.env.transform
[CONFIG] MinIO Endpoint: lakehouse-minio:9000
[CONFIG] Access Key (masked): tra***
DEBUG: Access Key is tra + ***
DEBUG: ENDPOINT is lakehouse-minio:9000


In [3]:
spark= create_spark_session("notebook_test")
Spain_23= spark.read.parquet("s3a://bronze/SPAIN/2023_BRONZE/")

2026-08-12 10:32:39 | INFO | lakehouse.__main__ | Creating Spark Session: notebook_test


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/12 10:34:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-08-12 10:36:10 | INFO | lakehouse.__main__ | Spark session created successfully


26/08/12 10:37:05 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
Spain_23.rdd.getNumPartitions()

4

In [5]:
Spain_23

DataFrame[beneficiario: string, grupo_empresa: string, provincia: string, municipio: string, medida: string, objetivo_esp: string, fec_ini: string, fec_fin: string, feaga: string, feader: string, importecofin: string, feader_cofin: string, importe_euros: string, source_country: string, source_year: string, ingested_at: timestamp]

In [5]:
Spain_23.printSchema()

root
 |-- beneficiario: string (nullable = true)
 |-- grupo_empresa: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- medida: string (nullable = true)
 |-- objetivo_esp: string (nullable = true)
 |-- fec_ini: string (nullable = true)
 |-- fec_fin: string (nullable = true)
 |-- feaga: string (nullable = true)
 |-- feader: string (nullable = true)
 |-- importecofin: string (nullable = true)
 |-- feader_cofin: string (nullable = true)
 |-- importe_euros: string (nullable = true)
 |-- source_country: string (nullable = true)
 |-- source_year: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [6]:
# let's know the number of non null or nan values present in each column

# 1. Build expressions dynamically based on each column's specific data type
count_expressions = []


for col_name, col_type in Spain_23.dtypes:
    # Base condition: works for ALL data types (Timestamps, Strings, Ints, etc.)
    condition = F.col(col_name).isNotNull()

    # Only append the NaN check if the column is a floating-point numeric type
    if col_type in ("double", "float"):
        condition = condition & ~F.isnan(F.col(col_name))
    
    # Rule B: Catch empty text cells in string columns
    elif col_type == "string":
        condition = condition & ~(F.trim(F.col(col_name))).isin("", "N/A", "n/a", "NA", "na")
    
    # Aggregate using the safe conditional block
    count_expressions.append(F.count(F.when(condition, 1)).alias(col_name))

# 2. Run the single, optimized aggregation across the cluster
counts_row = Spain_23.select(count_expressions).first()

print(counts_row)


Row(beneficiario=1949486, grupo_empresa=176, provincia=1949486, municipio=1949486, medida=1949486, objetivo_esp=15344, fec_ini=0, fec_fin=0, feaga=1949486, feader=1949486, importecofin=1949486, feader_cofin=1949486, importe_euros=1949486, source_country=1949486, source_year=1949486, ingested_at=1949486)


In [7]:
# Count total rows in the DataFrame
total_rows = Spain_23.count()

# Print header
print(f'{"Column Name": <65} | {"Missing Percentage"}')
print("-" * 85)

# Calculate and print missing percentage for each column
for column, valid_count in counts_row.asDict().items():
    missing_percentage = ((total_rows - valid_count) / total_rows) * 100
    print(f"{column: <65} | {missing_percentage: >10.3f}%")


Column Name                                                       | Missing Percentage
-------------------------------------------------------------------------------------
beneficiario                                                      |      0.000%
grupo_empresa                                                     |     99.991%
provincia                                                         |      0.000%
municipio                                                         |      0.000%
medida                                                            |      0.000%
objetivo_esp                                                      |     99.213%
fec_ini                                                           |    100.000%
fec_fin                                                           |    100.000%
feaga                                                             |      0.000%
feader                                                            |      0.000%
importecofin               

In [6]:
Spain_23_cleaned= Spain_23.select(
    F.col("beneficiario").alias("beneficiary"),
    F.col("municipio").alias("municipality"),
    F.col("provincia").alias("province"),
    F.col("source_country").alias("country"),
    F.col("source_year").alias("year"),
    F.col("medida").alias("intervention_code"),
    F.regexp_replace(F.col("feaga"), ",", ".").cast(DoubleType()).alias("total_eagf_income_support"),
    F.regexp_replace(F.col("feader"), ",", ".").cast(DoubleType()).alias("total_eafrd_income_support"),
    F.regexp_replace(F.col("importecofin"), ",", ".").cast(DoubleType()).alias("national_cofunding_amount")
    
).fillna(0.0, subset=["total_eagf_income_support", "total_eafrd_income_support", "national_cofunding_amount"])

# fill nulls in text fields
Spain23_cleaned= Spain_23_cleaned.fillna("UNKNOWN", subset= ["beneficiary", "municipality", "intervention_code"])

In [9]:
Spain23_cleaned.show(5, truncate=False)

+--------------+-------------------+--------+-------+----+----------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+--------------------------+-------------------------+
|beneficiary   |municipality       |province|country|year|intervention_code                                                                                                                                   |total_eagf_income_support|total_eafrd_income_support|national_cofunding_amount|
+--------------+-------------------+--------+-------+----+----------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+--------------------------+-------------------------+
|A BOUCIÑA S.C.|15911 - Rois       |A Coruña|SPAIN  |2023|II.1   Régimen de pago básico                                                    

In [11]:
spark.stop()